# Fine-tune a Small LLM on Wikipedia Data

This notebook demonstrates how to fine-tune a small language model (like Llama-3-8b or Gemma) using the dataset generated by your Wikipedia scraper.

## Prerequisites
1. Upload your `dataset.jsonl` file to the Colab runtime (Files sidebar -> Upload).
2. Select a GPU runtime (Runtime -> Change runtime type -> T4 GPU).

In [ ]:
%%capture
# Install Unsloth (Efficient Fine-tuning)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

In [ ]:
from unsloth import FastLanguageModel
import torch
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset

## 1. Load Model

In [ ]:
max_seq_length = 2048
dtype = None # None for auto detection
load_in_4bit = True # Use 4bit quantization to reduce memory usage

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit", # You can choose other models like unsloth/gemma-7b-bnb-4bit
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth", 
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

## 2. Load Dataset

In [ ]:
dataset_path = "dataset.jsonl"

# Load the dataset
dataset = load_dataset("json", data_files=dataset_path, split="train")

# Format the dataset for instruction tuning
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    outputs      = examples["response"]
    texts = []
    for instruction, output in zip(instructions, outputs):
        text = f"### Instruction:\n{instruction}\n\n### Response:\n{output}"
        texts.append(text)
    return { "text" : texts, }

dataset = dataset.map(formatting_prompts_func, batched = True)

## 3. Train

In [ ]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # Adjust this for longer training
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer.train()

## 4. Inference

In [ ]:
FastLanguageModel.for_inference(model)
inputs = tokenizer(
[
    "### Instruction:\nTell me about the root system of an oak tree.\n\n### Response:\n"
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 128, use_cache = True)
tokenizer.batch_decode(outputs)